# How to Build an Agent with TextualMOC

This is the **third tutorial** in the TextualMOC series. 

While [Tutorial 1](https://github.com/ggreco77/TextualMOC/tree/main/tuto1_TextualMOC) introduced the basics of creating and handling Textual MOCs, the [Tutorial 2](https://github.com/ggreco77/TextualMOC/tree/main/tuto2_SemanticMOC) the **semantic** dimension — transforming textual content into vector embeddings for Generative AI (GenAI) applications; Here, we explore how to build an agent using the [`LangGraph`](https://docs.langchain.com/oss/python/langgraph/overview) and [`LangChain`](https://docs.langchain.com/oss/python/langchain/overview) frameworks.

The notebook is associated with the paper **[Encapsulating Textual Contents into a MOC data structure for Advanced Applications](https://www.sciencedirect.com/science/article/pii/S2213133725000873)**; Greco et al., 2026.

### LLM Configuration

The LLM we will use is `qwen3:14b`, which can be installed locally with `Ollama`.

```bash
ollama pull qwen3:14b

In [ ]:
## 1)Setup

#conda create -n acme_tuto_vo_genai python=3.12 -y
#conda activate acme_tuto_vo_genai

#python -m pip install --upgrade pip setuptools wheel

#python -m pip install \
#  mocpy==0.20.0 \
#  matplotlib==3.10.8 \
#  numpy==2.4.3 \
#  requests==2.32.5 \
#  beautifulsoup4 \
#  ipyaladin==0.3.0 \
#  ipywidgets==8.1.8 \
#  astropy==7.2.0 \
#  pandas==3.0.1 \
#  langchain-community==0.4.1 \
#  langchain-ollama==1.0.1 \
#  scikit-learn \
#  astroquery==0.4.12.dev10525 \
#  cdshealpix==0.8.1 \
#  notebook==7.5.5 \
#  ipykernel==7.2.0 \
#  transformers==5.3.0

#python -m ipykernel install --user \
#  --name acme_tuto_vo_genai \
#  --display-name "Python (acme_tuto_vo_genai)"

##  Import and COnfiguration

While waiting for a stable library, the cell below downloads the `enriched_moc.py` module from GitHub, if it is not already present, and imports the `TextualMOC` class.

This module extends ordinary MOCs by encapsulating textual content, multimedia links, images, metadata, and cell annotations.


In [ ]:
import requests
from pathlib import Path

REQUIRED_VERSION = "0.0.7"
url = "https://raw.githubusercontent.com/ggreco77/TextualMOC/main/textualmoc/enriched_moc.py"
dest = Path("enriched_moc.py")

def get_local_version():
    if not dest.exists(): return None
    for line in dest.read_text().split('\n'):
        if '__version__' in line: return line.split('"')[1]
    return "unknown"

local = get_local_version()
if local is None:
    r = requests.get(url, timeout=30); r.raise_for_status()
    dest.write_bytes(r.content)
    print(f"Downloaded enriched_moc.py v{REQUIRED_VERSION}")
elif local == REQUIRED_VERSION:
    print(f"enriched_moc.py v{local} already up to date.")
else:
    print(f"Local version: {local} — required: {REQUIRED_VERSION}")

from enriched_moc import TextualMOC, SemanticMOC

In [ ]:
from __future__ import annotations

import copy
import difflib
import html
import json
from pathlib import Path
from typing import Any
import math

from typing_extensions import Annotated
from operator import add

import requests
import numpy as np
import ipywidgets as widgets
from IPython.display import HTML, Markdown, clear_output, display
from pydantic import BaseModel, Field, ValidationError

from mocpy import MOC
from enriched_moc import TextualMOC

from langchain.agents import AgentState, create_agent
from langchain.agents.middleware import wrap_tool_call
from langchain.messages import ToolMessage

from langchain.tools import ToolRuntime, tool
from langchain_ollama import ChatOllama
from langchain_core.messages import ToolMessage
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command, Overwrite


### Loading the TextualMOC Data

Before building the agent, we define the main configuration values and load the input `TextualMOC` file.

In this step, we specify the input and output paths, select the local Ollama model that will be used by the agent, configure a few execution options, and verify that the source JSON file is available before loading it into memory.

In [ ]:
# Path to the input TextualMOC JSON file.
TEXTUAL_MOC_JSON_PATH = Path("NGC4993.json")

# Path where the final enriched TextualMOC will be saved.
OUTPUT_JSON_PATH = Path("NGC4993_textual_moc_final.json")

# Local Ollama model used by the agent.
# This tutorial uses Qwen3 14B as the default LLM.
OLLAMA_MODEL = "qwen3:14b"

# Default thread identifier for the agent conversation/state.
DEFAULT_THREAD_ID = "textual-moc-ngc4993-thread-1"

# If True, automatically render the updated result after each turn.
AUTO_RENDER_AFTER_TURN = True

# If True, display the full JSON structure after each turn.
# Keeping this False makes notebook output easier to read.
SHOW_FULL_JSON_AFTER_TURN = False

# Make sure the input file exists before trying to load it.
if not TEXTUAL_MOC_JSON_PATH.exists():
    raise FileNotFoundError(
        f"Cannot find {TEXTUAL_MOC_JSON_PATH!s}. "
        "Place the JSON file in the same folder as the notebook or update the path."
    )

# Load the raw TextualMOC content from disk.
with TEXTUAL_MOC_JSON_PATH.open("r", encoding="utf-8") as f:
    RAW_TEXTUAL_MOC = json.load(f)

# Print the top-level keys to quickly inspect the loaded structure.
print("Loaded TextualMOC keys:", list(RAW_TEXTUAL_MOC.keys()))



### Helper Functions

The following helper functions support the main workflow by handling small, reusable tasks such as data access, formatting, validation, or rendering.

Keeping these utilities separate makes the notebook easier to read, reuse, and maintain.

In [ ]:
def extract_coverage_json(textual_moc_json: dict[str, Any]) -> dict[str, Any]:
    """Extract only the numeric MOC coverage keys from the canonical JSON."""
    return {k: v for k, v in textual_moc_json.items() if str(k).isdigit()}

def build_textual_moc(textual_moc_json: dict[str, Any]) -> TextualMOC:
    """Rebuild a TextualMOC object from the full JSON snapshot."""
    coverage_json = extract_coverage_json(textual_moc_json)
    moc = MOC.from_json(coverage_json)
    tm = TextualMOC(moc)
    tm.moc_data = copy.deepcopy(textual_moc_json)
    return tm

def snapshot_to_json_text(snapshot: dict[str, Any]) -> str:
    """Convert a snapshot dictionary into formatted JSON text."""
    return json.dumps(snapshot, ensure_ascii=False, indent=2)

def extract_cells(snapshot: dict[str, Any]) -> set[tuple[int, int]]:
    """Extract all `(order, pixel)` cells from a snapshot."""
    cells: set[tuple[int, int]] = set()
    for order, pixels in snapshot.items():
        if str(order).isdigit():
            for pixel in pixels:
                cells.add((int(order), int(pixel)))
    return cells

def build_moc_from_cells(cells: set[tuple[int, int]]) -> MOC | None:
    """Build a MOC object from a set of HEALPix cells."""
    if not cells:
        return None
    ordered = sorted(cells)
    pixels = np.array([pixel for _, pixel in ordered], dtype=np.uint64)
    depths = np.array([depth for depth, _ in ordered], dtype=np.uint8)
    max_depth = int(depths.max())
    return MOC.from_healpix_cells(pixels, depths, max_depth)

def compute_textual_moc_diff(
    previous: dict[str, Any] | None,
    current: dict[str, Any],
) -> dict[str, Any]:
    """Compute structural and metadata differences between two snapshots."""
    previous = previous or {}

    prev_cells = extract_cells(previous)
    curr_cells = extract_cells(current)

    changed_fields = {}
    tracked_fields = [
        "text",
        "multimedia",
        "image",
        "author",
        "date",
        "last_text_update",
        "annotated_cells",
    ]
    for field in tracked_fields:
        if previous.get(field) != current.get(field):
            changed_fields[field] = {
                "before": previous.get(field),
                "after": current.get(field),
            }

    added_orders = sorted(
        {k for k in current.keys() if str(k).isdigit()} -
        {k for k in previous.keys() if str(k).isdigit()},
        key=int,
    )
    removed_orders = sorted(
        {k for k in previous.keys() if str(k).isdigit()} -
        {k for k in current.keys() if str(k).isdigit()},
        key=int,
    )

    added_cells = sorted(curr_cells - prev_cells)
    removed_cells = sorted(prev_cells - curr_cells)

    return {
        "has_changes": bool(changed_fields or added_orders or removed_orders or added_cells or removed_cells),
        "changed_fields": changed_fields,
        "added_orders": added_orders,
        "removed_orders": removed_orders,
        "added_cells": added_cells,
        "removed_cells": removed_cells,
        "added_cells_count": len(added_cells),
        "removed_cells_count": len(removed_cells),
    }

def short_json(value: Any, max_len: int = 500) -> str:
    """Return a shortened string representation of a JSON-like value."""
    text = json.dumps(value, ensure_ascii=False, indent=2) if isinstance(value, (dict, list)) else str(value)
    return text if len(text) <= max_len else text[:max_len] + " …"



FULL_SKY_AREA_DEG2 = 41252.96124941927
def compute_moc_area_metrics(moc) -> dict[str, float]:
    sky_fraction = float(moc.sky_fraction)
    return {
        "sky_fraction": sky_fraction,
        "area_deg2": FULL_SKY_AREA_DEG2 * sky_fraction,
    }

def render_diff_panel(previous: dict[str, Any] | None, current: dict[str, Any]) -> None:
    """Render a visual summary of differences between two TextualMOC states."""
    diff = compute_textual_moc_diff(previous, current)

    if previous is None:
        display(Markdown("### Diff\nStato iniziale caricato: nessun confronto precedente disponibile."))
        return

    if not diff["has_changes"]:
        display(Markdown("### Diff\nNessuna modifica rilevata nel Textual MOC."))
        return

    cards = []
    for field, payload in diff["changed_fields"].items():
        cards.append(f"""
        <div style="border:1px solid #ddd; border-radius:10px; padding:12px; margin:8px 0;">
          <div style="font-weight:700; margin-bottom:8px;">Changed field: <code>{html.escape(field)}</code></div>
          <div style="display:grid; grid-template-columns: 1fr 1fr; gap: 12px;">
            <div>
              <div style="font-weight:600; color:#a33;">Before</div>
              <pre style="white-space:pre-wrap;">{html.escape(short_json(payload["before"]))}</pre>
            </div>
            <div>
              <div style="font-weight:600; color:#1a7f37;">After</div>
              <pre style="white-space:pre-wrap; background:#eefbf1; padding:6px; border-radius:8px;">{html.escape(short_json(payload["after"]))}</pre>
            </div>
          </div>
        </div>
        """)

    summary_html = f"""
    <h3>Diff del Textual MOC</h3>
    <ul>
      <li><b>Added orders</b>: {html.escape(str(diff["added_orders"]))}</li>
      <li><b>Removed orders</b>: {html.escape(str(diff["removed_orders"]))}</li>
      <li><b>Added cells</b>: {diff["added_cells_count"]}</li>
      <li><b>Removed cells</b>: {diff["removed_cells_count"]}</li>
    </ul>
    {''.join(cards)}
    """
    display(HTML(summary_html))

    if "text" in diff["changed_fields"]:
        before_text = (diff["changed_fields"]["text"]["before"] or "").splitlines()
        after_text = (diff["changed_fields"]["text"]["after"] or "").splitlines()
        html_table = difflib.HtmlDiff(wrapcolumn=100).make_table(
            before_text,
            after_text,
            fromdesc="Previous text",
            todesc="Updated text",
            context=True,
            numlines=1,
        )
        display(HTML("<h4>Diff testuale</h4>" + html_table))

RENDER_OUT = widgets.Output()
DIFF_OUT = widgets.Output()
JSON_OUT = widgets.Output()

#display(widgets.VBox([RENDER_OUT, DIFF_OUT, JSON_OUT]))


### Rendering the Current TextualMOC State

This function renders the current `TextualMOC` in the notebook and optionally compares it with a previous snapshot.

It is responsible for displaying the sky coverage, highlighting added and removed cells, showing a readable diff summary, and optionally printing the full JSON state. In practice, this is the main visualization helper used to inspect how the `TextualMOC` evolves after each interaction.


In [ ]:
def render_textual_moc_state(
    current_snapshot: dict[str, Any],
    previous_snapshot: dict[str, Any] | None = None,
    show_full_json: bool = False,
) -> None:
    """Render the current TextualMOC state and highlight changes from the previous snapshot."""
    tm = build_textual_moc(current_snapshot)

    # Primo render: non mostrare overlay di diff
    is_initial_render = previous_snapshot is None

    if is_initial_render:
        diff = {
            "added_cells": [],
            "removed_cells": [],
            "summary": "Primo render: nessuna differenza da mostrare.",
        }
    else:
        diff = compute_textual_moc_diff(previous_snapshot, current_snapshot)

    with RENDER_OUT:
        clear_output(wait=True)
        tm.render_ipyaladin(
            aladin_instance=None,
            color="magenta",
            opacity=0.35,
            survey="P/DSS2/color",
            fov=0.4,
            annotation_color="yellow",
            annotation_opacity=0.95,
        )

        viewer = getattr(tm, "ipyaladin", None)
        if viewer is not None and not is_initial_render:
            added_moc = build_moc_from_cells(set(diff["added_cells"]))
            removed_moc = build_moc_from_cells(set(diff["removed_cells"]))

            if added_moc is not None:
                viewer.add_moc(
                    added_moc,
                    name="added_cells",
                    color="lime",
                    opacity=0.9,
                    fill=True,
                    edge=True,
                )

            if removed_moc is not None:
                viewer.add_moc(
                    removed_moc,
                    name="removed_cells",
                    color="red",
                    opacity=0.95,
                    fill=False,
                    edge=True,
                )

        display(Markdown(
            "**Overlay legend** — magenta: current coverage; "
            "yellow: annotated cells; green: added cells; red: removed cells."
        ))

    with DIFF_OUT:
        clear_output(wait=True)
        if is_initial_render:
            display(Markdown("**Diff iniziale:** nessuna differenza da mostrare."))
        else:
            render_diff_panel(previous_snapshot, current_snapshot)

    with JSON_OUT:
        clear_output(wait=True)
        if show_full_json:
            print(snapshot_to_json_text(current_snapshot))

### Defining the Agent State

This section defines the shared state used by the agent during execution.

The state stores the current `TextualMOC`, tracks pending patches produced during the workflow, records the latest detected change, and keeps the path of the most recently saved file. Centralizing these values in a dedicated state object makes the graph easier to manage and update across steps.

- `messages`: the conversation history maintained by the agent;
- `textual_moc`: the **complete JSON** representation of the Textual MOC;
- `last_change`: a summary of the latest detected or applied change;
- `last_saved_path`: the path of the most recently saved file.
- `pending_patches`: the list of patch operations waiting to be applied to the Textual MOC.

In [ ]:
class TextualMOCAgentState(AgentState):
    """Store the shared state used across the TextualMOC agent workflow."""
    textual_moc: dict[str, Any]
    pending_patches: Annotated[list[dict[str, Any]], add]
    last_change: dict[str, Any]
    last_saved_path: str


### Initializing the Agent State

This helper function creates the initial state of the agent from the loaded `TextualMOC` JSON.

It starts the workflow with an empty conversation history, a deep copy of the input `TextualMOC`, no pending patches, an initial change summary, and no saved output path yet.

The line below then uses this function to build the starting state that will be passed into the agent workflow.

In [ ]:
def make_initial_state(raw_textual_moc: dict[str, Any]) -> dict[str, Any]:
    """Create the initial agent state from the loaded TextualMOC JSON."""
    return {
        "messages": [],
        "textual_moc": copy.deepcopy(raw_textual_moc),
        "pending_patches": [],
        "last_change": {"summary": "Stato iniziale caricato da JSON."},
        "last_saved_path": "",
    }


INITIAL_STATE = make_initial_state(RAW_TEXTUAL_MOC)


### Helper Functions and State Utilities

This section collects a set of helper functions and state utilities used throughout the agent workflow.

These functions handle common tasks such as loading and rebuilding `TextualMOC` objects, combining text and annotations, preparing patch commands, comparing textual content, and consolidating pending updates into the canonical agent state.


In [ ]:
def state_to_textual_moc(state: dict[str, Any]) -> TextualMOC:
    """Convert the agent state into a TextualMOC object."""
    return build_textual_moc(copy.deepcopy(state["textual_moc"]))

def load_textual_moc_from_local_file(input_file_path: str | Path) -> TextualMOC:
    input_file_path = Path(input_file_path).expanduser()

    if not input_file_path.exists():
        raise FileNotFoundError(f"File non trovato: {input_file_path}")

    with input_file_path.open("r", encoding="utf-8") as f:
        raw_textual_moc = json.load(f)

    return build_textual_moc(raw_textual_moc)

def join_nonempty_text_parts(*parts: str) -> str:
    """Join non-empty text fragments into a single formatted block."""
    cleaned = [part.strip() for part in parts if isinstance(part, str) and part.strip()]
    return "\n\n---\n\n".join(cleaned)

def merge_annotated_cells(
    left_snapshot: dict[str, Any],
    right_snapshot: dict[str, Any],
) -> dict[str, Any]:
    """Merge annotated cells from two snapshots into a single mapping."""
    merged = copy.deepcopy(left_snapshot.get("annotated_cells", {}))

    for order, pixels_map in right_snapshot.get("annotated_cells", {}).items():
        merged.setdefault(order, {})
        for pixel, note in pixels_map.items():
            merged[order][pixel] = note

    return merged

def patch_already_pending(runtime: ToolRuntime, patch: dict[str, Any]) -> bool:
    """Return True if an identical patch is already pending in the current state."""
    state = getattr(runtime, "state", {}) or {}
    pending_patches = state.get("pending_patches", []) if isinstance(state, dict) else []
    return any(existing_patch == patch for existing_patch in pending_patches)

def build_patch_command(
    patch: dict[str, Any],
    runtime: ToolRuntime,
    message: str,
    duplicate_message: str | None = None,
) -> Command:
    if patch_already_pending(runtime, patch):
        return Command(
            update={
                "messages": [
                    ToolMessage(
                        content=duplicate_message or "An identical patch is already pending and will not be scheduled twice.",
                        tool_call_id=runtime.tool_call_id,
                    )
                ]
            }
        )

    return Command(
        update={
            "pending_patches": [patch],
            "messages": [
                ToolMessage(
                    content=message,
                    tool_call_id=runtime.tool_call_id,
                )
            ],
        }
    )

def build_error_command(
    runtime: ToolRuntime,
    message: str,
) -> Command:
    """Build a command that records an error message for the current tool call."""
    return Command(
        update={
            "messages": [
                ToolMessage(
                    content=message,
                    tool_call_id=runtime.tool_call_id,
                )
            ]
        }
    )


#TEXT_INTERSECTION_MODEL = "qwen3:14b"

#text_intersection_llm = ChatOllama(
#    model=TEXT_INTERSECTION_MODEL,
#    temperature=0,
#)

#class StructuredTextIntersection(BaseModel):
#    same_object: bool = False
#    shared_information: list[str] = Field(default_factory=list)
#    morphology_comparison: str = ""
#    distance_comparison: str = ""
#    possible_conflicts: list[str] = Field(default_factory=list)
#    only_in_a: list[str] = Field(default_factory=list)
#    only_in_b: list[str] = Field(default_factory=list)
#    intersection_text: str = ""

def extract_json_object_from_llm_output(raw_text: str) -> dict[str, Any]:
    """Extract a valid JSON object from the raw LLM output."""
    text = raw_text.strip()

    if text.startswith("```"):
        text = re.sub(r"^```(?:json)?\s*", "", text)
        text = re.sub(r"\s*```$", "", text)

    start = text.find("{")
    end = text.rfind("}")

    if start == -1 or end == -1 or end < start:
        raise ValueError("Il modello non ha restituito un JSON valido.")

    return json.loads(text[start:end + 1])

#def compute_structured_text_intersection_with_llm(
#    left_text: str,
#    right_text: str,
#) -> StructuredTextIntersection:
#    prompt = f"""
#You are comparing two astronomical descriptions.

#TEXT_A:
#{left_text}

#TEXT_B:
#{right_text}

#Return ONLY a valid JSON object with this exact structure:

#{{
#  "same_object": false,
#  "shared_information": ["..."],
#  "morphology_comparison": "...",
#  "distance_comparison": "...",
#  "possible_conflicts": ["..."],
#  "only_in_a": ["..."],
#  "only_in_b": ["..."],
#  "intersection_text": "..."
#}}

#Rules:
#- Be scientifically conservative.
#- Do NOT treat generic overlap such as "both are galaxies", "both mention distance", or "both are astronomical objects" as meaningful intersection by itself.
#- A meaningful intersection should contain only information that is truly compatible and specific.
#- Check explicitly whether the two texts refer to the same astronomical object or clearly different objects.
#- Compare morphology explicitly:
#  - examples: elliptical galaxy, spiral galaxy, starburst galaxy, interacting galaxies, kilonova host galaxy
#  - if morphology differs, state that clearly in "morphology_comparison" and include it in "possible_conflicts" when appropriate
#- Compare distance explicitly:
#  - if the distances are clearly different, state that in "distance_comparison"
#- "same_object" must be true only if the texts clearly refer to the same source/object/system.
#- "shared_information" must contain only precise compatible statements, not generic category overlap.
#- "possible_conflicts" must include differences in morphology, object identity, physical interpretation, or distance when these make the descriptions non-equivalent.
#- "intersection_text" must be:
#  - a short English sentence only if there is real, specific shared astronomical information
#  - otherwise an empty string
#- If the texts describe different galaxies with different morphology or context, prefer an empty "intersection_text".
#- Do not invent facts.
#- Do not output markdown.
#- Do not output any text outside the JSON.
#""".strip()

#    response = text_intersection_llm.invoke(prompt)

#    if isinstance(response.content, str):
#        raw_text = response.content
#    else:
#        raw_text = "".join(
#            block.get("text", "")
#            for block in response.content
#            if isinstance(block, dict)
#        )

#    data = extract_json_object_from_llm_output(raw_text)

#    try:
#        return StructuredTextIntersection.model_validate(data)
#    except ValidationError as e:
#        raise ValueError(
#            f"Invalid LLM output for structured intersection: {e}"
#        ) from e

def intersect_annotated_cells(
    left_snapshot: dict[str, Any],
    right_snapshot: dict[str, Any],
) -> dict[str, Any]:
    """Return the annotated cells shared by both snapshots."""
    result: dict[str, Any] = {}

    left_cells = left_snapshot.get("annotated_cells", {})
    right_cells = right_snapshot.get("annotated_cells", {})

    for order, left_pixels in left_cells.items():
        if order not in right_cells:
            continue

        for pixel, left_note in left_pixels.items():
            if pixel not in right_cells[order]:
                continue

            right_note = right_cells[order][pixel]
            if left_note == right_note:
                note = left_note
            else:
                note = f"{left_note}\n---\n{right_note}"

            result.setdefault(order, {})
            result[order][pixel] = note

    return result

def snapshot_coverage_is_empty(snapshot: dict[str, Any]) -> bool:
    """Check whether the coverage section of a snapshot is empty."""
    return len(extract_coverage_json(snapshot)) == 0

def consolidate_state_values(
    values: dict[str, Any],
    apply_side_effects: bool = True,
) -> dict[str, Any]:
    """Apply pending patches, update the TextualMOC snapshot, and return the consolidated state."""
    previous_snapshot = copy.deepcopy(values["textual_moc"])
    patches = copy.deepcopy(values.get("pending_patches", []))
    tm = build_textual_moc(previous_snapshot)
    last_saved_path = values.get("last_saved_path", "")

    for patch in patches:
        op = patch["op"]

        if op == "append_text":
            tm.update_text_inline(patch["new_text"])

        elif op == "update_multimedia":
            current_text = tm.moc_data.get("text", "")
            current_image = tm.moc_data.get("image", "")
            tm.add_text_media_image(
                text_file_path=current_text,
                multimedia_url=patch["multimedia_url"],
                image=current_image,
            )

        elif op == "update_image":
            current_text = tm.moc_data.get("text", "")
            current_media = tm.moc_data.get("multimedia", "")
            tm.add_text_media_image(
                text_file_path=current_text,
                multimedia_url=current_media,
                image=patch["image_url"],
            )

        elif op == "annotate_cell":
            tm.annotate_cell(
                order=patch["order"],
                pixel=patch["pixel"],
                text=patch["note"],
            )

        elif op == "update_metadata":
            tm.update_metadata(
                author=patch.get("author") or None,
                date=patch.get("date") or None,
            )

        elif op == "merge_textual_moc_from_file":
            incoming_tm = load_textual_moc_from_local_file(patch["input_file_path"])

            current_moc = tm.moc
            incoming_moc = incoming_tm.moc

            if current_moc is not None and incoming_moc is not None:
                merged_moc = current_moc.union(incoming_moc)
            elif current_moc is not None:
                merged_moc = current_moc
            elif incoming_moc is not None:
                merged_moc = incoming_moc
            else:
                merged_moc = None

            merged_tm = TextualMOC(merged_moc)

            if merged_moc is not None:
                merged_tm.moc_data = copy.deepcopy(merged_moc.serialize(format="json"))
            else:
                merged_tm.moc_data = {}

            merged_text = join_nonempty_text_parts(
                tm.moc_data.get("text", ""),
                incoming_tm.moc_data.get("text", ""),
            )
            merged_multimedia = tm.moc_data.get("multimedia") or incoming_tm.moc_data.get("multimedia", "")
            merged_image = tm.moc_data.get("image") or incoming_tm.moc_data.get("image", "")

            if merged_text or merged_multimedia or merged_image:
                merged_tm.add_text_media_image(
                    text_file_path=merged_text,
                    multimedia_url=merged_multimedia,
                    image=merged_image,
                )

            merged_annotations = merge_annotated_cells(tm.moc_data, incoming_tm.moc_data)
            if merged_annotations:
                merged_tm.moc_data["annotated_cells"] = merged_annotations

            merged_author = tm.moc_data.get("author") or incoming_tm.moc_data.get("author", "")
            merged_date = tm.moc_data.get("date") or incoming_tm.moc_data.get("date", "")
            if merged_author or merged_date:
                merged_tm.update_metadata(
                    author=merged_author or None,
                    date=merged_date or None,
                )

            merged_last_text_update = (
                tm.moc_data.get("last_text_update")
                or incoming_tm.moc_data.get("last_text_update")
            )
            if merged_last_text_update:
                merged_tm.moc_data["last_text_update"] = merged_last_text_update

            tm = merged_tm

        elif op == "intersect_textual_moc_with_file":
            incoming_tm = load_textual_moc_from_local_file(patch["input_file_path"])

            current_moc = tm.moc
            incoming_moc = incoming_tm.moc

            if current_moc is not None and incoming_moc is not None:
                intersected_moc = current_moc.intersection(incoming_moc)
            else:
                intersected_moc = None

            intersected_tm = TextualMOC(intersected_moc)

            if intersected_moc is not None:
                intersected_tm.moc_data = copy.deepcopy(intersected_moc.serialize(format="json"))
            else:
                intersected_tm.moc_data = {}

            left_text = tm.moc_data.get("text", "").strip()
            right_text = incoming_tm.moc_data.get("text", "").strip()

            if left_text and right_text:
                text_intersection = compute_structured_text_intersection_with_llm(
                    left_text,
                    right_text,
                ).intersection_text.strip()
            elif left_text == right_text:
                text_intersection = left_text
            else:
                text_intersection = ""

            intersected_multimedia = (
                tm.moc_data.get("multimedia", "")
                if tm.moc_data.get("multimedia", "") == incoming_tm.moc_data.get("multimedia", "")
                else ""
            )
            intersected_image = (
                tm.moc_data.get("image", "")
                if tm.moc_data.get("image", "") == incoming_tm.moc_data.get("image", "")
                else ""
            )

            if text_intersection or intersected_multimedia or intersected_image:
                intersected_tm.add_text_media_image(
                    text_file_path=text_intersection,
                    multimedia_url=intersected_multimedia,
                    image=intersected_image,
                )

            intersected_annotations = intersect_annotated_cells(tm.moc_data, incoming_tm.moc_data)
            if intersected_annotations:
                intersected_tm.moc_data["annotated_cells"] = intersected_annotations

            intersected_author = (
                tm.moc_data.get("author", "")
                if tm.moc_data.get("author", "") == incoming_tm.moc_data.get("author", "")
                else ""
            )
            intersected_date = (
                tm.moc_data.get("date", "")
                if tm.moc_data.get("date", "") == incoming_tm.moc_data.get("date", "")
                else ""
            )
            if intersected_author or intersected_date:
                intersected_tm.update_metadata(
                    author=intersected_author or None,
                    date=intersected_date or None,
                )

            intersected_last_text_update = (
                tm.moc_data.get("last_text_update", "")
                if tm.moc_data.get("last_text_update", "") == incoming_tm.moc_data.get("last_text_update", "")
                else ""
            )
            if intersected_last_text_update:
                intersected_tm.moc_data["last_text_update"] = intersected_last_text_update

            tm = intersected_tm

    if apply_side_effects:
        for patch in patches:
            if patch["op"] == "save_to_file":
                tm.save(patch["output_file_path"])
                last_saved_path = patch["output_file_path"]

    current_snapshot = copy.deepcopy(tm.moc_data)
    diff = compute_textual_moc_diff(previous_snapshot, current_snapshot)

    return {
        **values,
        "textual_moc": current_snapshot,
        "pending_patches": [],
        "last_change": diff,
        "last_saved_path": last_saved_path,
    }

### Agent Tools

The agent uses the following tools to modify and manage the `TextualMOC`:

| Tool | Description |
|---|---|
| `annotate_textual_moc_cell` | Adds a textual annotation to an existing HEALPix cell. |
| `append_text_to_textual_moc` | Appends new content to the `text` field of the current `TextualMOC`. |
| `get_current_textual_moc_sky_fraction` | Returns the sky fraction of the current TextualMOC spatial coverage, together with the corresponding sky area in steradians and square degrees. |
| `merge_textual_moc_from_local_file` | Loads a local `TextualMOC` and merges it with the current one, combining spatial coverage and textual content. |
| `save_textual_moc_to_file` | Saves the current `TextualMOC` state to a local file. |
| `update_image_url` | Updates the `image` field without changing any other part of the `TextualMOC`. |
| `update_multimedia_url` | Updates the `multimedia` field while preserving the rest of the `TextualMOC`. |
| `update_textual_moc_metadata` | Updates the `author` and `date` metadata fields. |

In [ ]:
@tool(parse_docstring=True)
def append_text_to_textual_moc(new_text: str, runtime: ToolRuntime) -> Command:
    """Append new content to the `text` field of the current TextualMOC.

    Use this tool when the user wants to add textual information without
    replacing the existing text.

    Args:
        new_text: Text to append to the current `text` field.
    """
    return build_patch_command(
        patch={
            "op": "append_text",
            "new_text": new_text,
        },
        runtime=runtime,
        message="Text added correctly to the Textual MOC.",
    )


@tool(parse_docstring=True)
def update_multimedia_url(multimedia_url: str, runtime: ToolRuntime) -> Command:
    """Update the `multimedia` field while preserving the rest of the TextualMOC.

    Use this tool when the user provides or replaces the multimedia link
    associated with the current TextualMOC.

    Args:
        multimedia_url: Multimedia URL to store in the `multimedia` field.
    """
    return build_patch_command(
        patch={
            "op": "update_multimedia",
            "multimedia_url": multimedia_url,
        },
        runtime=runtime,
        message=f"multimedia field updated to: {multimedia_url}",
    )


@tool(parse_docstring=True)
def update_image_url(image_url: str, runtime: ToolRuntime) -> Command:
    """Update the `image` field while preserving the rest of the TextualMOC.

    Use this tool when the user provides or replaces the representative image
    URL for the current TextualMOC.

    Args:
        image_url: Image URL to store in the `image` field.
    """
    return build_patch_command(
        patch={
            "op": "update_image",
            "image_url": image_url,
        },
        runtime=runtime,
        message=f"image field updated to: {image_url}",
    )


@tool(parse_docstring=True)
def annotate_textual_moc_cell(order: int, pixel: int, note: str, runtime: ToolRuntime) -> Command:
    """Add a textual annotation to an existing HEALPix cell in the current TextualMOC.

    Use this tool only when the target cell already exists in the current
    TextualMOC coverage.

    Args:
        order: HEALPix order of the cell to annotate.
        pixel: HEALPix pixel index of the cell to annotate.
        note: Textual note to attach to the selected cell.
    """
    tm = state_to_textual_moc(runtime.state)
    order_key = str(order)

    if order_key not in tm.moc_data or pixel not in tm.moc_data[order_key]:
        return build_error_command(
            runtime=runtime,
            message=(
                f"Annotation not applied: the combination "
                f"order={order} e pixel={pixel} does not exist in the Textual MOC."
            ),
        )

    return build_patch_command(
        patch={
            "op": "annotate_cell",
            "order": order,
            "pixel": pixel,
            "note": note,
        },
        runtime=runtime,
        message=f"Annotation added to the cell with {order}, pixel {pixel}.",
    )


@tool(parse_docstring=True)
def update_textual_moc_metadata(author: str = "", date: str = "", runtime: ToolRuntime = None) -> Command:
    """Update the `author` and/or `date` metadata fields of the current TextualMOC.

    Use this tool when the user wants to set or revise metadata without changing
    coverage, annotations, or media fields.

    Args:
        author: Author name to store in the `author` field. Leave empty to keep it unchanged.
        date: Date value to store in the `date` field. Leave empty to keep it unchanged.
    """
    fragments = []

    if author:
        fragments.append(f"author='{author}'")
    if date:
        fragments.append(f"date='{date}'")

    summary = ", ".join(fragments) if fragments else "no fields"

    return build_patch_command(
        patch={
            "op": "update_metadata",
            "author": author,
            "date": date,
        },
        runtime=runtime,
        message=f"Metadata updated: {summary}.",
    )


@tool(parse_docstring=True)
def save_textual_moc_to_file(output_file_path: str, runtime: ToolRuntime) -> Command:
    """Save the current TextualMOC state to a local file.

    Use this tool when the user explicitly asks to persist the current state on disk.

    Args:
        output_file_path: Destination path of the output JSON file.
    """
    return build_patch_command(
        patch={
            "op": "save_to_file",
            "output_file_path": output_file_path,
        },
        runtime=runtime,
        message=f"Salvataggio schedulato in: {output_file_path}",
    )


@tool(parse_docstring=True)
def merge_textual_moc_from_local_file(input_file_path: str, runtime: ToolRuntime) -> Command:
    """Merge the current TextualMOC with a local TextualMOC file.

    This tool combines both spatial coverage and textual information from the
    input file with the current TextualMOC.

    Args:
        input_file_path: Path to the local TextualMOC JSON file to merge.
    """
    try:
        normalized_path = Path(input_file_path).expanduser()
        _ = load_textual_moc_from_local_file(normalized_path)
    except Exception as e:
        return build_error_command(
            runtime=runtime,
            message=(
                f"Merge non applicato: impossibile caricare il Textual MOC "
                f"da '{input_file_path}'. Reason: {type(e).__name__}: {e}"
            ),
        )

    return build_patch_command(
        patch={
            "op": "merge_textual_moc_from_file",
            "input_file_path": str(normalized_path),
        },
        runtime=runtime,
        message=(
            f"Merge scheduled from local file: {normalized_path}. "
            "Spatial coverage and textual content will be merged."
        ),
        duplicate_message=(
            f"Merge already scheduled from local file: {normalized_path}. "
            "The same merge patch will not be added twice."
        ),
    )


@tool(parse_docstring=True)
def intersect_textual_moc_with_local_file(
    input_file_path: str,
    runtime: ToolRuntime,
) -> Command:
    """Intersect the current TextualMOC with a local TextualMOC file.

    This tool computes both spatial and textual intersection with the local
    input file and schedules the corresponding state update.

    Args:
        input_file_path: Path to the local TextualMOC JSON file to intersect.
    """
    try:
        normalized_path = Path(input_file_path).expanduser()
        _ = load_textual_moc_from_local_file(normalized_path)

        return build_patch_command(
            patch={
                "op": "intersect_textual_moc_with_file",
                "input_file_path": str(normalized_path),
            },
            runtime=runtime,
            message=(
                f"Intersection scheduled from local file: {normalized_path}. "
                "Spatial and textual intersection will be computed."
            ),
            duplicate_message=(
                f"Intersection already scheduled from local file: {normalized_path}. "
                "The same intersection patch will not be added twice."
            ),
        )

    except Exception as e:
        return build_error_command(
            runtime=runtime,
            message=(
                f"ntersection not applied: Unable to load Textual MOC. "
                f"da '{input_file_path}'. Reason: {type(e).__name__}: {e}"
            ),
        )

@tool
def get_current_textual_moc_sky_fraction(runtime: ToolRuntime) -> Command:
    """Restituisce la sky_fraction e l'area del MOC corrente senza modificare lo stato."""
    try:
        current_tm = state_to_textual_moc(runtime.state)
        current_moc = current_tm.moc

        if current_moc is None:
            return build_error_command(
                runtime=runtime,
                message="Impossibile calcolare la sky_fraction: il Textual MOC corrente non contiene una coverage spaziale valida.",
            )

        metrics = compute_moc_area_metrics(current_moc)

        return Command(
            update={
                "messages": [
                    ToolMessage(
                        content=json.dumps(
                            {
                                "sky_fraction": metrics["sky_fraction"],
                                "area_deg2": metrics["area_deg2"],
                            },
                            ensure_ascii=False,
                            indent=2,
                        ),
                        tool_call_id=runtime.tool_call_id,
                    )
                ]
            }
        )

    except Exception as e:
        return build_error_command(
            runtime=runtime,
            message=(
                "Calcolo della sky_fraction fallito. "
                f"Reason: {type(e).__name__}: {e}"
            ),
        )

### Defining the TextualMOC Agent

This section defines the actual agent that will manage the `TextualMOC`.

First, a system prompt is created to specify the agent’s behavior: it must use tools whenever a valid modification is requested, avoid inventing missing values, operate only on existing cells, handle partial failures gracefully, and always provide the final answer in Italian with a clear distinction between successful and failed updates.

Next, the local language model is initialized through `ChatOllama` using the selected model configuration. An `InMemorySaver` checkpointer is also created to preserve the agent state across interactions.

A custom middleware function, `continue_on_tool_error`, is then defined with `@wrap_tool_call`. Its role is to prevent the whole agent run from failing when a single tool raises an exception. Instead of stopping execution, it returns a `ToolMessage` describing the error, so the agent can continue with the remaining valid tool calls and report a partial success.

Finally, `create_agent(...)` assembles the full agent by combining:
- the LLM,
- the available tools,
- the system prompt,
- the custom state schema,
- the in-memory checkpointer,
- and the middleware for robust error handling.

The result is a `TextualMOC` agent that can update the map, save it, merge external local files, and respond in a controlled and fault-tolerant way.


In [ ]:
SYSTEM_PROMPT = """
You are an assistant that modifies an astronomical Textual MOC (Multi-Order Coverage) map.

Operational instructions:

- Use tools whenever the user request requires modifying, annotating, merging, intersecting, or saving the Textual MOC, and whenever a suitable tool exists.
- Prefer tool-based execution over free-text explanations whenever the requested action is actually executable with the available tools.
- Do not invent missing data under any circumstances: never create orders, pixels, cells, file paths, parameters, metadata values, URLs, or any other values that were not explicitly provided and cannot be verified.
- Modify or annotate only cells that actually exist in the current Textual MOC.
- When the user asks to save, use the appropriate save tool, if available.
- If a request requires multiple operations, execute all valid ones.
- If one tool fails, do not stop the entire request: continue with the remaining valid tool calls.
- If one requested item does not exist, report that specific sub-task as failed, without treating it as a global fatal error.
- Never simulate executions that cannot actually be performed with the available tools.
- Never claim that an action was completed unless it was actually executed through a tool or clearly scheduled through the corresponding tool mechanism.
- If the user asks to load a Textual MOC from a local file and merge it with the current one, use the dedicated tool. A bare uploaded filename such as `M82.json` is a valid local path. Never invent a filename or path that the user did not provide.
- If the user asks for an operation that matches an available tool, call the tool instead of only describing what should be done.
- If no valid tool exists for a requested action, explicitly say so and explain that the action cannot be executed because of the current tool limitations.
- In such cases, also state which kind of tool would be needed to complete the request, if that can be determined from the request.
- If the user asks for the sky fraction or sky area of the current Textual MOC, use the dedicated tool.
- If a tool would be required but one or more mandatory parameters are missing, do not invent them.
- In that case, explain precisely:
  1) which tool would be needed,
  2) which required parameters are missing,
  3) why the action cannot be executed without them.
- If only part of the request is actionable, execute the actionable part with tools and clearly report the remaining part as not executable.
- Do not perform unsupported transformations implicitly. If there is no tool for a requested transformation, say that explicitly.
- Base every action on the current Textual MOC state and on the actual capabilities of the available tools.

Final answer requirements:

- Always separate clearly:
  1) successful updates
  2) failed or non-executable updates, with the reason
- Be explicit about partial success whenever only some sub-tasks were completed.
"""

llm = ChatOllama(
    model=OLLAMA_MODEL,
    temperature=0,
)

checkpointer = InMemorySaver()

@wrap_tool_call
def continue_on_tool_error(request, handler):
    """
    If a tool call fails, do not interrupt the whole agent run.
    Return an error ToolMessage instead, so that the remaining tool calls can continue
    and the model can report partial success in the final response.
    """
    try:
        return handler(request)
    except Exception as e:
        tool_name = request.tool_call.get("name", "unknown_tool")
        args = request.tool_call.get("args", {})
        return ToolMessage(
            content=(
                f"Tool '{tool_name}' failed.\n"
                f"Reason: {type(e).__name__}: {e}\n"
                f"Arguments: {args}\n"
                "Continue with the other successful tool calls and report partial success."
            ),
            tool_call_id=request.tool_call["id"],
        )

agent = create_agent(
    model=llm,
    tools=[
        append_text_to_textual_moc,
        update_multimedia_url,
        update_image_url,
        annotate_textual_moc_cell,
        update_textual_moc_metadata,
        save_textual_moc_to_file,
        merge_textual_moc_from_local_file,
        #intersect_textual_moc_with_local_file,
        get_current_textual_moc_sky_fraction,
    ],
    system_prompt=SYSTEM_PROMPT,
    state_schema=TextualMOCAgentState,
    checkpointer=checkpointer,
    middleware=[continue_on_tool_error],
    name="textual_moc_first_agent",
)


### Thread Management and Interactive Execution

This section defines the helper functions used to manage the agent state across conversation threads and to interact with the agent from the notebook.

It includes utilities to build the thread configuration, retrieve the current state, display approval buttons for pending updates, and send a user request to the agent. Together, these functions make it possible to run the agent step by step, inspect the proposed changes, and decide whether to apply or discard them before updating the canonical `TextualMOC` state.

- `get_thread_config(...)`: builds the configuration dictionary used to identify the current conversation thread.
- `get_current_state(...)`: retrieves the current agent state for the selected thread, if available.
- `show_update_state_buttons(...)`: displays interactive buttons to approve or discard pending state updates.
- `ask_agent(...)`: sends a user message to the agent, streams intermediate outputs, and manages rendering plus optional human approval of pending patches.


In [ ]:
def get_thread_config(thread_id: str = DEFAULT_THREAD_ID) -> dict[str, Any]:
    return {"configurable": {"thread_id": thread_id}}


def get_current_state(thread_id: str = DEFAULT_THREAD_ID) -> dict[str, Any] | None:
    snapshot = agent.get_state(get_thread_config(thread_id))
    values = getattr(snapshot, "values", None)
    if values and "textual_moc" in values:
        return values
    return None


def show_update_state_buttons(
    thread_id: str,
    previous_snapshot: dict[str, Any],
    auto_render: bool,
    show_full_json: bool,
) -> None:
    config = get_thread_config(thread_id)

    approve_btn = widgets.Button(
        description="Update state",
        button_style="success",
        icon="check",
    )
    reject_btn = widgets.Button(
        description="Cancel",
        button_style="danger",
        icon="times",
    )

    status_out = widgets.Output()
    title = widgets.HTML("<b>Do you confirm the state update?</b>")

    def disable_buttons():
        approve_btn.disabled = True
        reject_btn.disabled = True

    def on_approve(_):
        print("CLICK ON 'Update state''")

        live_state = get_current_state(thread_id)
        if live_state is None:
            with status_out:
                clear_output(wait=True)
                display(Markdown("**❌ No state available to update.**"))
            disable_buttons()
            return

        approved_state = consolidate_state_values(
            live_state,
            apply_side_effects=True,
        )

        agent.update_state(
            config,
            values={
                "textual_moc": approved_state["textual_moc"],
                "pending_patches": Overwrite([]),
                "last_change": approved_state["last_change"],
                "last_saved_path": approved_state["last_saved_path"],
            },
            as_node="tools",
        )

        updated_state = get_current_state(thread_id)

        with status_out:
            clear_output(wait=True)
            display(Markdown("**✅ State updated.**"))

        if auto_render and updated_state is not None:
            render_textual_moc_state(
                current_snapshot=updated_state["textual_moc"],
                previous_snapshot=previous_snapshot,
                show_full_json=show_full_json,
            )

        disable_buttons()

    def on_reject(_):
        print("CLICK ON 'Cancel''")

        agent.update_state(
            config,
            values={
                "pending_patches": Overwrite([]),
            },
            as_node="tools",
        )

        reverted_state = get_current_state(thread_id)

        with status_out:
            clear_output(wait=True)
            display(Markdown("**❌ Update cancelled. Pending patches have been discarded."))

        if auto_render and reverted_state is not None:
            render_textual_moc_state(
                current_snapshot=reverted_state["textual_moc"],
                previous_snapshot=previous_snapshot,
                show_full_json=show_full_json,
            )

        disable_buttons()

    approve_btn.on_click(on_approve)
    reject_btn.on_click(on_reject)

    display(
        widgets.VBox([
            title,
            widgets.HBox([approve_btn, reject_btn]),
            status_out,
        ])
    )


def ask_agent(
    user_message: str,
    thread_id: str = DEFAULT_THREAD_ID,
    auto_render: bool = AUTO_RENDER_AFTER_TURN,
    show_full_json: bool = SHOW_FULL_JSON_AFTER_TURN,
) -> dict[str, Any]:
    print(f"\n🧑 You: {user_message}\n")

    config = get_thread_config(thread_id)
    existing_state = get_current_state(thread_id)

    if existing_state is not None:
        previous_snapshot = copy.deepcopy(existing_state["textual_moc"])
        payload = {
            "messages": [{"role": "user", "content": user_message}],
        }
    else:
        previous_snapshot = copy.deepcopy(INITIAL_STATE["textual_moc"])
        payload = {
            **copy.deepcopy(INITIAL_STATE),
            "messages": [{"role": "user", "content": user_message}],
        }

    for chunk in agent.stream(payload, config=config, stream_mode="updates"):
        data = chunk.get("data", chunk) if isinstance(chunk, dict) else chunk

        if not isinstance(data, dict):
            continue

        for node_name, node_output in data.items():
            if not isinstance(node_output, dict):
                continue

            messages = node_output.get("messages", [])
            for msg in messages:
                tool_calls = getattr(msg, "tool_calls", None)
                content = getattr(msg, "content", None)

                if tool_calls:
                    for call in tool_calls:
                        print(f"🔧 TOOL → {call['name']}({call['args']})")
                elif content:
                    prefix = "🤖 Agente" if node_name != "tools" else "✅ Tool result"
                    print(f"{prefix}: {content}")

    current_state = get_current_state(thread_id)

    if current_state is None:
        raise RuntimeError("No state available after the run.")

    if current_state.get("pending_patches"):
        candidate_state = consolidate_state_values(
            current_state,
            apply_side_effects=False,
        )

        if auto_render:
            render_textual_moc_state(
                current_snapshot=candidate_state["textual_moc"],
                previous_snapshot=previous_snapshot,
                show_full_json=show_full_json,
            )

        show_update_state_buttons(
            thread_id=thread_id,
            previous_snapshot=previous_snapshot,
            auto_render=auto_render,
            show_full_json=show_full_json,
        )

        return {
            "status": "waiting_human_approval",
            "thread_id": thread_id,
            "candidate_state": candidate_state,
            "current_state": current_state,
        }

    if auto_render:
        render_textual_moc_state(
            current_snapshot=current_state["textual_moc"],
            previous_snapshot=previous_snapshot,
            show_full_json=show_full_json,
        )

    return current_state

### Initial Viewer Setup

This final setup step displays the main output areas of the notebook and renders the initial `TextualMOC` state.

Three dedicated sections are shown: the interactive viewer, the diff panel, and the JSON output. The last call performs the first render of the initial snapshot, without any previous state to compare against.

In [ ]:
# Display the main section title for the interactive TextualMOC viewer.
display(Markdown("## Viewer Textual MOC"))
display(RENDER_OUT)

# Display the section that will show differences between states.
display(Markdown("## Diff"))
display(DIFF_OUT)

# Display the section reserved for the full JSON output, when enabled.
display(Markdown("## JSON"))
display(JSON_OUT)

# Render the initial TextualMOC snapshot.
# Since this is the first render, there is no previous snapshot to compare with.
#render_textual_moc_state
render_textual_moc_state(
    current_snapshot=copy.deepcopy(INITIAL_STATE["textual_moc"]),
    previous_snapshot=None,
    show_full_json=False,
)

## 11) Usage Examples

The following examples show how to interact with the agent in practice.

Each call:
- uses the LangGraph thread;
- updates the agent state;
- re-renders the full `TextualMOC` in `ipyaladin`;
- shows the highlighted diff between the previous and updated state.

In [ ]:
state = ask_agent("""
Add the following text: The spectral series evolves rapidly, showing sub-hour variations 
at early times and indicating the early presence of strontium, consistent with theory. 

Also add a new image link: https://www.ligo.caltech.edu/page/press-release-gw170817 
and a new multimedia link: https://www.media.inaf.it/2018/11/15/cosa-e-successo-alla-kilonova-gw170817/ 

Update the author to Giuseppe Greco. 

Also annotate the cell with order=10 and pixel=60400.

Remove the term 'NGC 4993' from the text."""
)

In [ ]:
state = ask_agent(
    "annotate the cell in the current textual moc with order=10; pixel=6373777, 'Gal' "
)


In [ ]:
ask_agent(
    "mesure the area of the current textual MOC"
)

In [ ]:
ask_agent(
    "Save the current textual MOC in 'test.json'"
)

## Checking the current state


In [ ]:
current_state = get_current_state()
print(snapshot_to_json_text(current_state["textual_moc"]))
